[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/92_assignment_regularized_logistic_regression.ipynb)

# Assignment 92 — Regularized Logistic Regression

Nhiệm vụ: dự đoán các **vi mạch** sản xuất tại nhà máy có vượt qua kiểm định chất lượng hay không, dựa trên kết quả 2 bài test (bài tập kinh điển của khoá Andrew Ng). Học trước bài [03](03_regularized_logistic_regression.ipynb).

### Chú ý
Dùng **advanced optimization** (`scipy.optimize.minimize`); bạn cung cấp hàm cost và gradient **có regularization**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit as sigmoid          # sigmoid ổn định số học của scipy
from sklearn.preprocessing import PolynomialFeatures

from ml_course.checks import check_close, check_greater, check_shape, check_true
from ml_course.utils import numerical_gradient, relative_error, repo_path

## 1. Trực quan hoá dữ liệu

In [ ]:
data = np.loadtxt(repo_path("data", "sample", "ex4_microchip_tests.txt"), delimiter=",")
X, y = data[:, :2], data[:, 2]


def plot_data(X, y, ax=None):
    ax = ax or plt.gca()
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=30, c="darkblue", marker="+", label="Accepted")
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=30, c="gold", marker="o", edgecolors="k", linewidths=0.4, label="Rejected")
    ax.set_xlabel("Microchip Test 1")
    ax.set_ylabel("Microchip Test 2")
    ax.legend(loc="upper right")


plot_data(X, y)
plt.show()

Không có đường thẳng nào tách được hai nhóm → cần ranh giới **phi tuyến**.

## 2. Feature mapping

Dùng `PolynomialFeatures(6)` ánh xạ $(x_1, x_2)$ thành mọi đơn thức bậc ≤ 6: $1, x_1, x_2, x_1^2, x_1x_2, x_2^2, \dots, x_2^6$ — **28 feature** (kể cả cột hằng số 1).

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
poly = None
X2 = None
### END CODE HERE ###
print(X2.shape)

In [ ]:
check_shape(X2, (118, 28))

## 3. Cost function và gradient có regularization

$$J(\theta) = -\frac{1}{m}\left[y^\top\log(h) + (1 - y)^\top\log(1 - h + \varepsilon)\right] + \frac{\lambda}{2m}\sum_{j=1}^{n}\theta_j^2$$

**Không phạt $\theta_0$.**

In [ ]:
def cost_function_reg(theta, X, y, lam):
    m = len(y)
    eps = 1e-15
    ### START CODE HERE ### (≈ 3 dòng)
    h = None
    J = None
    ### END CODE HERE ###
    return J

In [ ]:
check_close(cost_function_reg(np.zeros(28), X2, y, 1.0), 0.6931471805599454, tol=1e-6, message="J(θ=0, λ=1) = ln 2")
check_close(cost_function_reg(np.ones(28), X2, y, 10.0), 3.1645093316150095, tol=1e-6, message="J(θ=1, λ=10)")

**Đầu ra kỳ vọng:** `J(θ = 0) = 0.693`, `J(θ = 1, λ = 10) = 3.16`

**Gradient:**

$$\nabla J(\theta) = \frac{1}{m}\left[X^\top\left(h - y\right) + \lambda\theta'\right]$$

trong đó $\theta'$ là $\theta$ với phần tử đầu bằng 0 (không phạt $\theta_0$). *Gợi ý:* `np.r_[0, theta[1:]]` hoặc `np.insert`.

> 📝 *Đính chính so với bản gốc:* số hạng regularization trong gradient là $\frac{\lambda}{m}\theta'$ (bản gốc thiếu hệ số $\frac1m$).

In [ ]:
def gradient_func_reg(theta, X, y, lam):
    m = len(y)
    ### START CODE HERE ### (≈ 3 dòng)
    h = None
    theta_reg = None
    grad = None
    ### END CODE HERE ###
    return grad

In [ ]:
t = np.random.default_rng(0).normal(size=28)
num = numerical_gradient(lambda th: cost_function_reg(th, X2, y, 3.0), t.copy())
check_true(relative_error(gradient_func_reg(t, X2, y, 3.0), num) < 1e-6, "gradient check (λ = 3)")

## 4. Tìm θ và vẽ decision boundary

Tối ưu với λ = 1:

In [ ]:
def fit(lam):
    res = minimize(cost_function_reg, np.zeros(X2.shape[1]), args=(X2, y, lam), jac=gradient_func_reg, method="BFGS")
    return res.x


### START CODE HERE ### (≈ 2 dòng)
theta = None
accuracy = None
### END CODE HERE ###
print(f"Accuracy train (λ = 1): {accuracy:.2%}")

In [ ]:
check_greater(accuracy, 0.80, "accuracy train > 80%")

**Đầu ra kỳ vọng:** khoảng `83%`

Decision boundary là đường đồng mức $\theta^\top\phi(x) = 0$ — vẽ bằng `contour` trên lưới điểm:

In [ ]:
def plot_boundary(theta, ax, title):
    u = np.linspace(-1, 1.5, 120)
    U, V = np.meshgrid(u, u)
    Z = (poly.transform(np.c_[U.ravel(), V.ravel()]) @ theta).reshape(U.shape)
    plot_data(X, y, ax)
    ax.contour(U, V, Z, levels=[0], colors="green", linewidths=2)
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(6, 5))
plot_boundary(theta, ax, "λ = 1")
plt.show()

## 5. Thay đổi giá trị của λ

**Bài tập:** fit với λ = 0, 1, 100 và vẽ 3 decision boundary cạnh nhau. Quan sát: λ = 0 → overfit (ranh giới uốn éo bám từng điểm); λ = 100 → underfit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
accs = {}
for ax, lam in zip(axes, [0.0, 1.0, 100.0]):
    ### START CODE HERE ### (≈ 2 dòng)
    th = None
    ### END CODE HERE ###
    plot_boundary(th, ax, f"λ = {lam}: accuracy train = {accs[lam]:.1%}")
plt.show()

In [ ]:
check_true(accs[0.0] >= accs[1.0] >= accs[100.0], "λ càng lớn, accuracy TRAIN càng giảm (model đơn giản hơn)")

### Tổng kết

- Feature mapping đa thức cho ranh giới phi tuyến; regularization kiểm soát overfitting.
- λ = 0: ranh giới phức tạp quá mức (overfit); λ quá lớn: underfit. Chọn λ bằng cross-validation (chương 15).

---
<sub>Bài gốc: *Programming Assignment: Regularized Logistic Regression* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License), dựa trên bài tập của Andrew Ng. Đã đính chính hệ số λ/m trong gradient, thêm kiểm tra tự động và gradient check.</sub>